In [0]:
# Config & Setup
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType
)
from pyspark.sql.functions import current_timestamp

CATALOG = 'fahrlytics'
BRONZE  = f'{CATALOG}.bronze'
BASE_PATH = f'/Volumes/{CATALOG}/bronze/raw_data'

TRIPS_PATH       = f'{BASE_PATH}/trips/'
CLICKSTREAM_PATH = f'{BASE_PATH}/clickstream/'

# Stable paths for Auto Loader state tracking
TRIPS_CHECKPOINT       = f'{BASE_PATH}/_checkpoints/trips'
CLICKSTREAM_CHECKPOINT = f'{BASE_PATH}/_checkpoints/clickstream'
TRIPS_SCHEMA_LOC       = f'{BASE_PATH}/_schema/trips'
CLICKSTREAM_SCHEMA_LOC = f'{BASE_PATH}/_schema/clickstream'

spark.sql(f'USE CATALOG {CATALOG}')

In [0]:
# Explicit Schema Hints (Faster startup, prevents type drift)
trips_schema = StructType([
    StructField('trip_id',          StringType(),  True),
    StructField('customer_id',      StringType(),  True),
    StructField('driver_id',        StringType(),  True),
    StructField('city',             StringType(),  True),
    StructField('pickup_time',      StringType(),  True), 
    StructField('drop_time',        StringType(),  True),
    StructField('distance_km',      DoubleType(),  True),
    StructField('fare_amount',      DoubleType(),  True),
    StructField('currency',         StringType(),  True),
    StructField('surge_multiplier', DoubleType(),  True),
    StructField('trip_status',      StringType(),  True),
    StructField('vehicle_type',     StringType(),  True),
    StructField('weather',          StringType(),  True),
    StructField('is_ghost_trip',    BooleanType(), True),
    StructField('is_fare_inflated', BooleanType(), True),
    StructField('day_type',         StringType(),  True),
])

clickstream_schema = StructType([
    StructField('event_id',            StringType(), True),
    StructField('customer_id',         StringType(), True),
    StructField('session_id',          StringType(), True),
    StructField('event_type',          StringType(), True),
    StructField('event_timestamp',     StringType(), True),
    StructField('city',                StringType(), True),
    StructField('device_type',         StringType(), True),
    StructField('ab_test_group',       StringType(), True),
    StructField('ab_test_experiment',  StringType(), True),
])

In [0]:
# Ensure Target Delta Tables Exist
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.trips (
        trip_id          STRING,
        customer_id      STRING,
        driver_id        STRING,
        city             STRING,
        pickup_time      STRING,
        drop_time        STRING,
        distance_km      DOUBLE,
        fare_amount      DOUBLE,
        currency         STRING,
        surge_multiplier DOUBLE,
        trip_status      STRING,
        vehicle_type     STRING,
        weather          STRING,
        is_ghost_trip    BOOLEAN,
        is_fare_inflated BOOLEAN,
        day_type         STRING,
        _ingested_at     TIMESTAMP
    ) USING DELTA
    COMMENT 'Raw trip events ingested via Auto Loader'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.clickstream (
        event_id           STRING,
        customer_id        STRING,
        session_id         STRING,
        event_type         STRING,
        event_timestamp    STRING,
        city               STRING,
        device_type        STRING,
        ab_test_group      STRING,
        ab_test_experiment STRING,
        _ingested_at       TIMESTAMP
    ) USING DELTA
    COMMENT 'Raw telemetry clickstream events ingested via Auto Loader'
""")

In [0]:
# Auto Loader for Trips
(
    spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.schemaLocation', TRIPS_SCHEMA_LOC)
        .schema(trips_schema)
        .load(TRIPS_PATH)
        .withColumn('_ingested_at', current_timestamp())
    .writeStream
        .format('delta')
        .option('checkpointLocation', TRIPS_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(f'{BRONZE}.trips')
        .awaitTermination()
)
print("Bronze Trips loaded successfully via Auto Loader.")

In [0]:
# Auto Loader for Clickstream
(
    spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.schemaLocation', CLICKSTREAM_SCHEMA_LOC)
        .schema(clickstream_schema)
        .load(CLICKSTREAM_PATH)
        .withColumn('_ingested_at', current_timestamp())
    .writeStream
        .format('delta')
        .option('checkpointLocation', CLICKSTREAM_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(f'{BRONZE}.clickstream')
        .awaitTermination()
)
print("Bronze Clickstream loaded successfully via Auto Loader.")